In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_dir = Path.cwd().resolve()
FILE_PATH = current_dir / "data" / "knapPI_1_1000_1000.csv"

print(FILE_PATH.exists())

file = pd.read_csv(FILE_PATH, sep=";", header=None) 

True


In [2]:
capacity = file.iloc[0, 1]

print(capacity)

5002


In [3]:
df = file.iloc[1:].reset_index(drop=True)

In [4]:
c = df.iloc[:, 0].astype(int).to_numpy()
w = df.iloc[:, 1].astype(int).to_numpy()

### Итак, что мы имеем?  

#### Множества и индексы  
$C$ - вместимость рюкзака  
$I \ = \ <p_i, w_i> \ = \ <p, w>_i \ \in \mathbb{N}^2$ - множество предметов (двумерный вектор, состоящий из ценности и веса $i$ предмета)  
$X \in \mathbb{B}^1$ - бинарный вектор, каждый $x_i$ элемент которого отвечает на вопрос: "Берем ли мы предмет, или нет?"   

#### Ограничения
$$\sum_{i \in I}w_i \cdot x_i \leq C$$

#### Целевая функция
$$z = \sum_{i \in I}c_i \cdot x_i \rightarrow \max$$

Реализуем два алгоритма - алгоритм **динамического программирования** и $G^{\frac{3}{4}}$.

#### Алгоритм динамического программирования

Обозначим через $f_k(R)$ - максимальную суммарную ценность от первых $k$ предметов при доступном остаточном весе $R = <r_1, ..., r_T>$  
(R - Remained weight)
Тогда:  

$$\begin{cases} f_0(R) = 0 \\ f_{k+1}(R) = \max(f_k(R), r_{k+1} + f_{k}(R - w_{k+1})), k=0,...,N-1, 0\leq Y \leq R \end{cases}$$

$f_k(R)$ = максимальная ценность от первых $k$ предметов при остаточном весе $R$

In [5]:
C = capacity
n = len(c)

dp = np.zeros((n + 1, C + 1), dtype=np.int64)

for i in range(1, n + 1):
    ci = int(c[i - 1])
    wi = int(w[i - 1])
    dp[i, :] = dp[i - 1, :]
    for R in range(wi, C + 1):
        dp[i, R] = max(dp[i, R], ci + dp[i - 1, R - wi])

best_value = dp[n, C]
print("Максимальная ценность:", best_value)

R = C
chosen = []

for i in range(n, 0, -1):
    if dp[i, R] != dp[i - 1, R]:
        chosen.append(i - 1)  
        R -= int(w[i - 1])

chosen.reverse()

print("Выбранные индексы:", chosen)
print("Суммарная ценность:", sum(int(c[j]) for j in chosen))
print("Суммарный вес:", sum(int(w[j]) for j in chosen))

Максимальная ценность: 54503
Выбранные индексы: [6, 10, 12, 13, 23, 25, 32, 35, 37, 38, 48, 53, 60, 121, 134, 137, 146, 147, 151, 216, 236, 245, 249, 254, 269, 273, 281, 334, 347, 362, 373, 379, 382, 419, 421, 426, 446, 463, 469, 473, 476, 480, 493, 494, 539, 573, 585, 592, 598, 599, 603, 610, 612, 657, 669, 703, 708, 718, 732, 736, 737, 743, 751, 770, 775, 786, 822, 824, 830, 845, 849, 855, 886, 887, 914, 937, 945, 967, 984, 986, 987, 989, 992]
Суммарная ценность: 54503
Суммарный вес: 5002


#### Алгоритм $G^\frac{3}{4}$

итак, обозначим за $I' \subseteq I$ подмножество предметов, $R'$ - остаточная вместимость.  

Упорядочим $I'$ по убыванию удельной ценности:  
$$\frac{c_{i_1}}{w_{i_1}} \geq \frac{c_{i_2}}{w_{i_2}} \geq ... \geq \frac{c_{i_m}}{w_{i_m}}$$

Тогда просто жадное решение это (просто берем по порядку предметы из подмножества, пока сможем брать по весу): $$z^G(I', R') = \sum_{i \in I', \sum_{w_i} \leq R'} c_i$$

Тогда улучшенное жадное решение $$z^{MG}(I', R') = \max(z^G(I',R'), \max_{i \in I', w_i \leq R'} p_i)$$

Теперь возьмем начальную оценку: $$z^A = \max_{i \in I}c_i$$  

И для каждой упорядоченной пары $$<q,k> \ \in I \times I, q \neq k$$

Проверяем условие $$w_q + w_k \leq R$$

И если оно выполнено, то формируем подзадачу  

$$\begin{cases} I_{q,k} = \{j \in I | c_j \leq \min(c_q, c_k)\} \setminus {q,k} \\
R_{q,k} = R - w_q - w_k \end{cases}$$

К этой задаче применяем улучшенный жадный алгоритм и обновляем оценку  
$$Z^{MG}_{q,k} = z^{MG}(I_{q,k}, R_{q,k}) \Longrightarrow z^{A} = \max(z^{A}, c_q + c_k + z_{q,k}^{MG})$$

In [ ]:
def amg(subset, capacity, c, w):

    if not subset or capacity <= 0:
        return 0

    # сортируем предметы по удельной ценности по убыванию
    items = [(c[i], w[i], i) for i in subset]
    items.sort(key=lambda x: x[0] / x[1] if x[1] > 0 else 0, reverse=True)

    z_G = 0
    rem_cap = capacity
    max_p_fit = 0

    for ci, wi, idx in items:
        if wi <= rem_cap:
            z_G += ci
            rem_cap -= wi
        if wi <= capacity and ci > max_c_fit:
            max_c_fit = ci

    return max(z_G, max_c_fit)

In [ ]:
def g34(c, w, C):

    n = len(c)
    if n == 0:
        return 0

    order = list(range(n))
    order.sort(key=lambda i: c[i] / w[i] if w[i] > 0 else 0, reverse=True)

    # z^A := max{p_j}
    z_A = max(c)

    # перебор всех пар (i, k), i != k
    for i in range(n):
        for k in range(n):
            if i == k:
                continue
            if w[i] + w[k] > C:
                continue

            M = min(c[i], c[k])
            cap = C - w[i] - w[k]

            # формируем подмножество: p_j <= M, j != i, j != k
            # применяем AMG
            z_G = 0
            rem_cap = cap
            max_c_fit = 0

            for j in order:
                if j == i or j == k:
                    continue
                if c[j] > M:
                    continue
                if w[j] <= rem_cap:
                    z_G += c[j]
                    rem_cap -= w[j]
                if w[j] <= cap and c[j] > max_c_fit:
                    max_c_fit = c[j]

            z_MG = max(z_G, max_c_fit)

            # обновляем лучшую оценку
            if c[i] + c[k] + z_MG > z_A:
                z_A = c[i] + c[k] + z_MG

    return z_A

In [ ]:
z_approx = g34(c, w, C)
print("Приближённое решение G^{3/4}:", z_approx)